# Exploração — Track 2: Water Quality Inversion

Notebook de desenvolvimento do pipeline de predição de turbidez e clorofila-a a partir de imagens Sentinel-2.  
Cobre desde a inspeção dos dados brutos até o treino final do GradientBoostingRegressor e validação da inferência local.

**Estrutura do repo (executar a partir da raiz `water-quality-inversion/`):**
```
water-quality-inversion/
  src/                     ← código do pipeline
  models/                  ← modelos .joblib treinados
  track2_download_link_*/  ← dados locais (não versionados)
  test_output/             ← saída da inferência local
```

In [ ]:
# Célula 0 — Setup: adiciona src/ ao path e limpa cache de módulos
# Deve ser executada primeiro sempre que o kernel for reiniciado.
import sys, pathlib, shutil, os

# raiz do projeto = pasta onde este notebook está
PROJECT_ROOT = pathlib.Path(os.getcwd())
SRC_PATH     = str(PROJECT_ROOT / 'src')

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# limpa .pyc e cache de módulos para garantir recarregamento limpo
src_dir = PROJECT_ROOT / 'src'
for pyc in src_dir.rglob('*.pyc'):
    pyc.unlink()
if (src_dir / '__pycache__').exists():
    shutil.rmtree(src_dir / '__pycache__')
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['train', 'model', 'dataset', 'build_records', 'infer']):
        del sys.modules[mod]

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'src no path  : {SRC_PATH in sys.path}')

In [ ]:
# Célula 1 — Estrutura de arquivos do projeto
# Exibe a árvore de diretórios ignorando dados pesados e cache.
import os
from pathlib import Path

IGNORAR = {'__pycache__', '.git', '.venv', 'track2_download_link_1',
           'track2_download_link_2', 'track2_download_link_3',
           'track2_download_link_4', 'track2_download_link_5'}

for root, dirs, files in os.walk('.'):
    dirs[:] = [d for d in sorted(dirs) if not d.startswith('.') and d not in IGNORAR]
    level  = root.replace('.', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root) or "."}/'
          if os.path.basename(root) else f'{indent}./')
    for f in sorted(files):
        fpath    = os.path.join(root, f)
        size     = os.path.getsize(fpath)
        size_str = f'{size/1024/1024:.1f}MB' if size > 1024*1024 else f'{size/1024:.0f}KB'
        print(f'{indent}  {f}  ({size_str})')

In [ ]:
# Célula 2 — Inspeção dos TIFs e CSVs
# Verifica metadados de dois TIFs (um válido, um minúsculo) e mostra
# as primeiras linhas dos CSVs de treino para confirmar formato.
import rasterio
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path('track2_download_link_2')

for tif_path in [
    DATA / 'area6/area6_images/area6_2024-02-04.tif',  # TIF grande (~209MB)
    DATA / 'area6/area6_images/area6_2024-02-09.tif',  # TIF minúsculo (81x76)
]:
    with rasterio.open(tif_path) as src:
        print(f'\n{tif_path.name}')
        print(f'  bandas : {src.count}')
        print(f'  shape  : {src.shape}  (height x width)')
        print(f'  CRS    : {src.crs}')
        print(f'  res    : {src.res}')
        print(f'  bounds : {src.bounds}')
        b1 = src.read(1).astype(float)
        print(f'  min/max banda1: {np.nanmin(b1):.4f} / {np.nanmax(b1):.4f}')

for csv_path in [
    DATA / 'area6/track2_turb_train_point_area6.csv',
    DATA / 'area6/track2_cha_train_point_area6.csv',
    DATA / 'area7/track2_turb_train_point_area7.csv',
]:
    df  = pd.read_csv(csv_path)
    col = [c for c in df.columns if c not in ('filename', 'Lon', 'Lat')][0]
    print(f'\n{csv_path.name}')
    print(df.head(3).to_string())
    print(f'  shape: {df.shape}  |  {col}: min={df[col].min():.2f}  max={df[col].max():.2f}  mean={df[col].mean():.2f}')

In [ ]:
# Célula 3 — Extração de patch 11x11
# Demonstra o alinhamento entre coordenadas lat/lon do CSV e pixels do TIF,
# e mostra os valores espectrais das 12 bandas no pixel central.
import rasterio
import numpy as np
import pandas as pd
from pathlib import Path

TIF = Path('track2_download_link_2/area6/area6_images/area6_2024-02-04.tif')
CSV = Path('track2_download_link_2/area6/track2_turb_train_point_area6.csv')

PATCH = 11
HALF  = PATCH // 2

with rasterio.open(TIF) as src:
    df  = pd.read_csv(CSV)
    row = df[df['filename'] == TIF.name].iloc[0]
    lon, lat = row['Lon'], row['Lat']

    py, px = src.index(lon, lat)
    print(f'Ponto: lon={lon:.6f}  lat={lat:.6f}')
    print(f'Pixel: row={py}  col={px}  |  shape imagem: {src.shape}')

    window = rasterio.windows.Window(px - HALF, py - HALF, PATCH, PATCH)
    patch  = src.read(window=window).astype(np.float32)  # (12, 11, 11)

    print(f'\nShape do patch: {patch.shape}')
    print(f'Valores das 12 bandas no pixel central:')
    print(np.round(patch[:, HALF, HALF], 4))
    print(f'\nSTD espacial (variabilidade no patch 11x11):')
    print(np.round(patch.reshape(12, -1).std(-1), 4))

In [ ]:
# Célula 4 — EDA: contagem de amostras, distribuição dos targets, TIFs minúsculos
# Mostra quantas amostras existem por área, a distribuição de turbidez e chl-a
# em escala original e log1p, e identifica TIFs inválidos (< 100x100 pixels).
import rasterio
import numpy as np
import pandas as pd
from pathlib import Path

AREAS = {
    'area1': {
        'img_dir': Path('track2_download_link_5/area1/area1_images'),
        'turb':    Path('track2_download_link_5/area1/track2_turb_train_point_area1.csv'),
        'cha':     Path('track2_download_link_5/area1/track2_cha_train_point_area1.csv'),
    },
    'area2': {
        'img_dir': Path('track2_download_link_4/area2/area2_images'),
        'turb':    Path('track2_download_link_4/area2/track2_turb_train_point_area2.csv'),
        'cha':     None,
    },
    'area3': {
        'img_dir': Path('track2_download_link_3/area3/area3_images'),
        'turb':    Path('track2_download_link_3/area3/track2_turb_train_point_area3.csv'),
        'cha':     None,
    },
    'area5': {
        'img_dir': Path('track2_download_link_3/area5/area5_images'),
        'turb':    Path('track2_download_link_3/area5/track2_turb_train_point_area5.csv'),
        'cha':     Path('track2_download_link_3/area5/track2_cha_train_point_area5.csv'),
    },
    'area6': {
        'img_dir': Path('track2_download_link_2/area6/area6_images'),
        'turb':    Path('track2_download_link_2/area6/track2_turb_train_point_area6.csv'),
        'cha':     Path('track2_download_link_2/area6/track2_cha_train_point_area6.csv'),
    },
    'area7': {
        'img_dir': Path('track2_download_link_2/area7/area7_images'),
        'turb':    Path('track2_download_link_2/area7/track2_turb_train_point_area7.csv'),
        'cha':     Path('track2_download_link_2/area7/track2_cha_train_point_area7.csv'),
    },
}

# --- contagem por área ---
print('=' * 50)
print('AMOSTRAS POR ÁREA E ALVO')
print('=' * 50)
total_turb, total_cha = 0, 0
for area, cfg in AREAS.items():
    n_turb = len(pd.read_csv(cfg['turb'])) if cfg['turb'] else 0
    n_cha  = len(pd.read_csv(cfg['cha']))  if cfg['cha']  else 0
    total_turb += n_turb
    total_cha  += n_cha
    print(f'  {area}: turb={n_turb:>4}  cha={n_cha:>4}')
print(f'  TOTAL : turb={total_turb:>4}  cha={total_cha:>4}')

# --- distribuição dos targets ---
print('\n' + '=' * 50)
print('DISTRIBUIÇÃO DOS TARGETS')
print('=' * 50)
for target in ['turb', 'cha']:
    vals = []
    for cfg in AREAS.values():
        if cfg[target]:
            col = 'turb_value' if target == 'turb' else 'cha_value'
            vals.extend(pd.read_csv(cfg[target])[col].dropna().tolist())
    vals = np.array(vals)
    print(f'\n  {target} (original): n={len(vals)}  min={vals.min():.2f}  max={vals.max():.2f}'
          f'  mean={vals.mean():.2f}  median={np.median(vals):.2f}  p99={np.quantile(vals, 0.99):.1f}')
    lv = np.log1p(vals)
    print(f'  {target} (log1p)  : min={lv.min():.3f}  max={lv.max():.3f}  mean={lv.mean():.3f}  std={lv.std():.3f}')

# --- TIFs minúsculos ---
print('\n' + '=' * 50)
print('TIFs MUITO PEQUENOS (shape < 100x100)')
print('=' * 50)
for area, cfg in AREAS.items():
    for tif in sorted(cfg['img_dir'].glob('*.tif')):
        with rasterio.open(tif) as src:
            h, w = src.shape
            if h < 100 or w < 100:
                print(f'  {area}/{tif.name}: {h}x{w}  → incluído em SKIP_TIFS')

In [ ]:
# Célula 5 — Construção dos records e validação do Dataset
# build_records carrega todos os patches válidos (12 bandas, 11x11) de cada área.
# WaterQualityDataset encapsula os records e aplica as transformações de features.
# A verificação de shape confirma que o vetor de features (29 dims) está correto.
from build_records import build_records
from dataset import WaterQualityDataset
import numpy as np
import torch

records_turb = build_records('turb')
records_cha  = build_records('cha')

ds_turb = WaterQualityDataset(records_turb, target='turb', augment=False)
ds_cha  = WaterQualityDataset(records_cha,  target='cha',  augment=False)

feat, label = ds_turb[0]
print(f'feature shape  : {feat.shape}')   # esperado: torch.Size([29])
print(f'feature range  : {feat.min():.3f} – {feat.max():.3f}')
print(f'label (log1p)  : {label.item():.3f}')

labels_turb = np.array([ds_turb[i][1].item() for i in range(len(ds_turb))])
labels_cha  = np.array([ds_cha[i][1].item()  for i in range(len(ds_cha))])
print(f'\nturb log1p — mean={labels_turb.mean():.3f}  std={labels_turb.std():.3f}'
      f'  min={labels_turb.min():.3f}  max={labels_turb.max():.3f}')
print(f'cha  log1p — mean={labels_cha.mean():.3f}  std={labels_cha.std():.3f}'
      f'  min={labels_cha.min():.3f}  max={labels_cha.max():.3f}')

In [ ]:
# Célula 6 — Análise geográfica: range lat/lon do treino vs área8
# Motivação para remover lat/lon das features: área8 (Oregon) está fora
# do range geográfico de todo o conjunto de treino (EUA central/leste).
# Incluir coordenadas faria o modelo extrapolar para fora da distribuição.
from build_records import build_records
import numpy as np

records = build_records('turb')
lats = np.array([r['lat'] for r in records])
lons = np.array([r['lon'] for r in records])

print('Range geográfico do conjunto de treino:')
print(f'  lat: {lats.min():.2f} – {lats.max():.2f}')
print(f'  lon: {lons.min():.2f} – {lons.max():.2f}')

print('\nRange geográfico de área8 (conjunto de teste):')
print('  lat: ~44.5 – 45.5  (Oregon, EUA)')
print('  lon: ~-122.8 – -122.2')
print('\n→ área8 está completamente fora do domínio de treino.')
print('  Decisão: remover lat/lon das features para evitar extrapolação geográfica.')

In [ ]:
# Célula 7 — Leave-one-area-out cross-validation (turbidez)
# Simula o cenário de generalização para uma área não vista no treino,
# que é exatamente o problema que enfrentamos com área8.
# R² negativo em várias áreas confirma que o modelo não generaliza bem
# para domínios espectrais diferentes — esperado com dados limitados.
from build_records import build_records
from train import build_features
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score

records = build_records('turb')

vals    = np.array([r['label'] for r in records])
cap     = np.quantile(vals, 0.99)
records = [r for r in records if r['label'] <= cap]

X, y  = build_features(records)
areas = np.array([r['area'] for r in records])

print('Leave-one-area-out R² (turbidez):')
for area in sorted(set(areas)):
    train_mask = areas != area
    val_mask   = areas == area
    if val_mask.sum() < 5:
        continue
    gbr = GradientBoostingRegressor(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, min_samples_leaf=5, random_state=42)
    gbr.fit(X[train_mask], y[train_mask])
    pred     = gbr.predict(X[val_mask])
    r2       = r2_score(y[val_mask], pred)
    rmse_orig = np.sqrt(np.mean((np.expm1(pred) - np.expm1(y[val_mask]))**2))
    print(f'  {area}: R²={r2:+.3f}  RMSE_orig={rmse_orig:.1f}  n={val_mask.sum()}')

In [ ]:
# Célula 8 — Medianas mensais para fallback
# Quando o patch é inválido (OOB, NaN, TIF ausente), o infer.py usa a mediana
# mensal do conjunto de treino como predição de fallback.
# Esta célula calcula e exibe os valores usados no infer.py.
from build_records import build_records
import numpy as np
from collections import defaultdict

records      = build_records('turb')
records_cha  = build_records('cha')

for target, recs in [('turb', records), ('cha', records_cha)]:
    por_mes = defaultdict(list)
    for r in recs:
        por_mes[r['month']].append(r['label'])
    print(f'\n{target} — mediana por mês (usada como fallback no infer.py):')
    for mes in sorted(por_mes):
        med = np.median(por_mes[mes])
        print(f'  mês {mes:>2}: n={len(por_mes[mes]):>3}  mediana={med:.2f}')
    print(f'  global : mediana={np.median([r["label"] for r in recs]):.2f}')

In [ ]:
# Célula 9 — Treino final do GradientBoostingRegressor
# Treina nos records completos (sem split de validação) e salva os modelos
# em models/model_turb.joblib e models/model_cha.joblib.
# CV R² negativo é esperado dado o domain shift entre áreas —
# o score real será revelado pela plataforma de avaliação com área8.
from build_records import build_records
from train import train_target
from pathlib import Path

Path('models').mkdir(exist_ok=True)

records_turb = build_records('turb')
records_cha  = build_records('cha')

r2_turb = train_target(records_turb, 'turb', 'models/model_turb.joblib')
r2_cha  = train_target(records_cha,  'cha',  'models/model_cha.joblib')

In [ ]:
# Célula 10 — Inferência local no sample de área8
# Roda o mesmo pipeline que o container Docker usa em produção,
# mas apontando para o sample local em track2_download_link_1/.
# Valida formato dos JSONs de saída antes de submeter ao GitLab.
import json
import numpy as np
import joblib
from pathlib import Path
from infer import extract_features, predict_csv

INPUT_DIR  = Path('track2_download_link_1/Guide to the Second Round_track2/test_input_sample')
OUTPUT_DIR = Path('test_output')
OUTPUT_DIR.mkdir(exist_ok=True)
IMG_DIR    = INPUT_DIR / 'area8_images'

model_turb = joblib.load('models/model_turb.joblib')
model_cha  = joblib.load('models/model_cha.joblib')

for target, csv_name, out_name in [
    ('turb', 'track2_turb_test_point.csv', 'result_turbidity.json'),
    ('cha',  'track2_cha_test_point.csv',  'result_chla.json'),
]:
    model  = model_turb if target == 'turb' else model_cha
    result = predict_csv(INPUT_DIR / csv_name, IMG_DIR, model, target)

    with open(OUTPUT_DIR / out_name, 'w') as f:
        json.dump(result, f, indent=2)

    vals = [v[0] for v in result.values()]
    print(f'{out_name}: {len(result)} pontos')
    print(f'  exemplo : {next(iter(result.items()))}')
    print(f'  range   : {min(vals):.4f} – {max(vals):.4f}')
    print(f'  mean    : {np.mean(vals):.4f}')